# Random Disclosure Parity Game: Gradient Bandit Agents

## Game Definition

> *The Random Disclosure Parity Game is a two-player zero-sum game with four moves:*
>
> 1. **Player I** selects an integer from $\{1, 2\}$.
> 2. **Referee** tosses a fair coin. Heads $\to$ Player II is informed of Player I's choice; Tails $\to$ Player II receives no information.
> 3. **Player II** selects an integer from $\{3, 4\}$.
> 4. **Referee** draws an integer from $\{1, 2, 3\}$ with probabilities $\{0.4, 0.2, 0.4\}$.
>
> The sum of moves 1, 3, and 4 is computed. If even, Player II pays Player I \$1; if odd, Player I pays Player II \$1.

In this notebook, we implement **Gradient Bandit** agents that learn to play through self-play. Unlike Q-learning which estimates action values, gradient bandits learn a *preference* $H(a)$ for each action and use softmax to convert preferences to probabilities:

$$\pi(a) = \frac{e^{H(a)}}{\sum_{b} e^{H(b)}}$$

This is a natural fit for the Parity Game, whose Nash equilibrium requires **mixed strategies** — gradient bandits learn stochastic policies directly.

### Known Equilibrium

| Quantity | Value |
|:---------|:------|
| Game value | $v = -0.30$ (Player II has a \$0.30 edge) |
| Player I | Indifferent: mix $\{1, 2\}$ with $\pi(1)=\pi(2)=0.5$ |
| Player II at $H_1$ | Pure: play 3 (match parity) |
| Player II at $H_2$ | Pure: play 4 (match parity) |
| Player II at $T$ | Mix: $\pi(3)=\pi(4)=0.5$ |

### Preference Decay for Adversarial Self-Play

In zero-sum games, standard gradient bandits can **diverge**: one player commits to an action, the other exploits it, causing preferences to spiral to $\pm\infty$. We add a small **preference decay** $\lambda$ that shrinks preferences toward zero after each update:

$$H(a) \leftarrow (1 - \lambda)\, H(a)$$

This prevents divergence while preserving the correct equilibrium structure: when one action truly dominates ($H_1$, $H_2$), the preference can still grow large enough for near-deterministic play; when both actions are equally good ($T$, PI), the decay pulls preferences back toward zero, yielding the equilibrium 50/50 mix.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from typing import Dict
import random
from tqdm import tqdm

plt.style.use('dark_background')
plt.rcParams['font.family'] = 'monospace'
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.facecolor'] = '#16213e'
plt.rcParams['figure.facecolor'] = '#0f0f23'
plt.rcParams['axes.edgecolor'] = '#4a4a6a'
plt.rcParams['axes.labelcolor'] = '#e94560'
plt.rcParams['xtick.color'] = '#a0a0a0'
plt.rcParams['ytick.color'] = '#a0a0a0'
plt.rcParams['text.color'] = '#e0e0e0'
plt.rcParams['grid.color'] = '#1a1a3a'
plt.rcParams['grid.alpha'] = 0.5

## 1. Game Environment

Player I sees nothing before choosing (single information set). Player II observes one of three information sets depending on the coin toss:

| Info Set | Coin | Disclosure |
|:--------:|:----:|:-----------|
| $H_1$ | Heads | PI chose 1 |
| $H_2$ | Heads | PI chose 2 |
| $T$   | Tails | PI's choice hidden |

In [ ]:
PI_ACTIONS = [1, 2]
PII_ACTIONS = [3, 4]
REFEREE_OUTCOMES = [(1, 0.4), (2, 0.2), (3, 0.4)]
COIN_PROB = 0.5


class ParityGame:
    """Random Disclosure Parity Game environment.

    Episode flow:
      1. PI picks i ∈ {1, 2}
      2. Coin flip → heads / tails
      3. PII picks j ∈ {3, 4} (sees i only if heads)
      4. Referee draws r ∈ {1, 2, 3} with P = {0.4, 0.2, 0.4}
      5. Payoff to PI: +1 if (i + j + r) even, −1 if odd
    """

    def reset(self):
        self.pi_choice = None
        self.coin = None
        self.pii_choice = None
        self.ref_draw = None
        self.done = False
        self.payoff_pi = 0.0
        return self

    def pi_step(self, i: int):
        """Player I picks i ∈ {1, 2}, then the coin is flipped."""
        self.pi_choice = i
        self.coin = 'H' if random.random() < COIN_PROB else 'T'

    def get_pii_info_set(self) -> str:
        """Return PII's information set: 'H1', 'H2', or 'T'."""
        if self.coin == 'H':
            return f'H{self.pi_choice}'
        return 'T'

    def pii_step(self, j: int):
        """PII picks j ∈ {3, 4}, referee draws, payoff is resolved."""
        self.pii_choice = j
        r = random.random()
        self.ref_draw = 1 if r < 0.4 else (2 if r < 0.6 else 3)
        total = self.pi_choice + self.pii_choice + self.ref_draw
        self.payoff_pi = 1.0 if total % 2 == 0 else -1.0
        self.done = True


game = ParityGame()
game.reset()
game.pi_step(1)
print(f"PI chose 1, coin = {game.coin}, PII info set = {game.get_pii_info_set()}")
game.pii_step(3)
print(f"PII chose 3, referee drew {game.ref_draw}, "
      f"sum = {game.pi_choice + game.pii_choice + game.ref_draw}, "
      f"payoff to PI = {game.payoff_pi:+.0f}")

## 2. Gradient Bandit Algorithm

The **Gradient Bandit** algorithm (Sutton & Barto, §2.8) learns numerical *preferences* $H(a)$ and selects actions via softmax:

$$\pi(a) = \frac{e^{H(a)}}{\sum_{b} e^{H(b)}}$$

After receiving reward $R$, preferences are updated by stochastic gradient ascent on $\mathbb{E}[R]$:

$$H(A) \leftarrow H(A) + \alpha \,(R - \bar{R})\,(1 - \pi(A))$$
$$H(a) \leftarrow H(a) - \alpha \,(R - \bar{R})\,\pi(a) \quad \text{for } a \neq A$$

followed by preference decay:

$$H(a) \leftarrow (1 - \lambda)\, H(a) \quad \forall a$$

where $\bar{R}$ is a running baseline (average reward) and $\lambda$ is the decay rate.

### Why gradient bandits suit this game

- The policy $\pi$ is **inherently stochastic** — it can represent mixed strategies directly.
- When two actions are equally good (PI's choice, PII at $T$), preferences stay near zero and softmax outputs ≈ 50/50.
- When one action dominates (PII at $H_1$, $H_2$), the preference diverges and softmax becomes nearly deterministic.
- The decay $\lambda$ stabilises training in adversarial self-play by preventing preference divergence.

In [ ]:
def softmax(h: np.ndarray) -> np.ndarray:
    """Numerically stable softmax."""
    e = np.exp(h - np.max(h))
    return e / e.sum()


class GradientBanditPI:
    """Gradient bandit agent for Player I (maximiser).

    Single decision point: preferences H(1), H(2).
    Maximises expected payoff to PI.
    """

    def __init__(self, alpha: float = 0.05, decay: float = 0.001,
                 use_baseline: bool = True):
        self.alpha = alpha
        self.decay = decay
        self.use_baseline = use_baseline
        self.H = np.zeros(len(PI_ACTIONS))
        self.avg_reward = 0.0
        self.n = 0

    def get_probs(self) -> dict:
        p = softmax(self.H)
        return dict(zip(PI_ACTIONS, p.tolist()))

    def select_action(self, training: bool = True) -> int:
        p = softmax(self.H)
        idx = np.random.choice(len(PI_ACTIONS), p=p)
        return PI_ACTIONS[idx]

    def update(self, action: int, reward: float):
        """Gradient bandit update (PI maximises reward)."""
        self.n += 1
        baseline = self.avg_reward if self.use_baseline else 0.0
        self.avg_reward += (reward - self.avg_reward) / self.n

        probs = softmax(self.H)
        advantage = reward - baseline
        idx = PI_ACTIONS.index(action)

        for i in range(len(PI_ACTIONS)):
            if i == idx:
                self.H[i] += self.alpha * advantage * (1 - probs[i])
            else:
                self.H[i] -= self.alpha * advantage * probs[i]

        self.H *= (1 - self.decay)


class GradientBanditPII:
    """Gradient bandit agent for Player II (minimiser).

    Three information sets: H1, H2, T.
    Separate preferences per info set.
    Minimises PI's payoff (maximises −payoff_PI).
    """

    INFO_SETS = ['H1', 'H2', 'T']

    def __init__(self, alpha: float = 0.05, decay: float = 0.001,
                 use_baseline: bool = True):
        self.alpha = alpha
        self.decay = decay
        self.use_baseline = use_baseline
        self.H = {s: np.zeros(len(PII_ACTIONS)) for s in self.INFO_SETS}
        self.avg_reward = {s: 0.0 for s in self.INFO_SETS}
        self.n = {s: 0 for s in self.INFO_SETS}

    def get_probs(self, info_set: str) -> dict:
        p = softmax(self.H[info_set])
        return dict(zip(PII_ACTIONS, p.tolist()))

    def select_action(self, info_set: str, training: bool = True) -> int:
        p = softmax(self.H[info_set])
        idx = np.random.choice(len(PII_ACTIONS), p=p)
        return PII_ACTIONS[idx]

    def update(self, info_set: str, action: int, reward_pi: float):
        """Gradient bandit update (PII minimises PI payoff → maximises −reward_pi)."""
        pii_reward = -reward_pi
        self.n[info_set] += 1
        baseline = self.avg_reward[info_set] if self.use_baseline else 0.0
        self.avg_reward[info_set] += (
            pii_reward - self.avg_reward[info_set]) / self.n[info_set]

        probs = softmax(self.H[info_set])
        advantage = pii_reward - baseline
        idx = PII_ACTIONS.index(action)

        for i in range(len(PII_ACTIONS)):
            if i == idx:
                self.H[info_set][i] += self.alpha * advantage * (1 - probs[i])
            else:
                self.H[info_set][i] -= self.alpha * advantage * probs[i]

        self.H[info_set] *= (1 - self.decay)

## 3. Training Loop

Each episode is a single game. Both agents update their preferences immediately with the terminal reward.

In [ ]:
def train_agents(
    pi_agent: GradientBanditPI,
    pii_agent: GradientBanditPII,
    num_episodes: int = 500_000,
    log_interval: int = 2000
) -> dict:
    """Train both agents through self-play."""
    game = ParityGame()
    stats = {
        'episodes': [],
        'avg_payoff_pi': [],
        'pi_prob_1': [], 'pi_prob_2': [],
        'pi_H1': [], 'pi_H2': [],
        'pii_H1_prob_3': [], 'pii_H2_prob_4': [], 'pii_T_prob_3': [],
        'pii_pref_H1': [], 'pii_pref_H2': [], 'pii_pref_T': [],
        'entropy_pi': [], 'entropy_pii_T': [],
    }
    payoff_buffer = []

    for ep in tqdm(range(num_episodes), desc='Training'):
        game.reset()
        pi_action = pi_agent.select_action(training=True)
        game.pi_step(pi_action)
        info_set = game.get_pii_info_set()
        pii_action = pii_agent.select_action(info_set, training=True)
        game.pii_step(pii_action)

        payoff_pi = game.payoff_pi
        payoff_buffer.append(payoff_pi)

        pi_agent.update(pi_action, payoff_pi)
        pii_agent.update(info_set, pii_action, payoff_pi)

        if (ep + 1) % log_interval == 0:
            stats['episodes'].append(ep + 1)
            stats['avg_payoff_pi'].append(
                np.mean(payoff_buffer[-log_interval:]))

            pi_p = pi_agent.get_probs()
            stats['pi_prob_1'].append(pi_p[1])
            stats['pi_prob_2'].append(pi_p[2])
            stats['pi_H1'].append(float(pi_agent.H[0]))
            stats['pi_H2'].append(float(pi_agent.H[1]))

            for s in ['H1', 'H2', 'T']:
                p = pii_agent.get_probs(s)
                if s == 'H1':
                    stats['pii_H1_prob_3'].append(p[3])
                    stats['pii_pref_H1'].append(
                        [float(pii_agent.H[s][0]),
                         float(pii_agent.H[s][1])])
                elif s == 'H2':
                    stats['pii_H2_prob_4'].append(p[4])
                    stats['pii_pref_H2'].append(
                        [float(pii_agent.H[s][0]),
                         float(pii_agent.H[s][1])])
                else:
                    stats['pii_T_prob_3'].append(p[3])
                    stats['pii_pref_T'].append(
                        [float(pii_agent.H[s][0]),
                         float(pii_agent.H[s][1])])

            def entropy(probs):
                return -sum(p * np.log(p + 1e-12) for p in probs)

            stats['entropy_pi'].append(
                entropy(list(pi_p.values())))
            stats['entropy_pii_T'].append(
                entropy(list(pii_agent.get_probs('T').values())))

    return stats

## 4. Training the Agents

In [ ]:
np.random.seed(42)
random.seed(42)

pi_agent = GradientBanditPI(alpha=0.05, decay=0.001, use_baseline=True)
pii_agent = GradientBanditPII(alpha=0.05, decay=0.001, use_baseline=True)

NUM_EPISODES = 500_000

print("Random Disclosure Parity Game — Gradient Bandit Agents")
print("═" * 60)
print(f"  Episodes:          {NUM_EPISODES:,}")
print(f"  Learning rate α:   {pi_agent.alpha}")
print(f"  Preference decay λ: {pi_agent.decay}")
print(f"  Baseline:          {pi_agent.use_baseline}")
print(f"  PI actions:        {PI_ACTIONS}")
print(f"  PII actions:       {PII_ACTIONS}")
print(f"  P(heads):          {COIN_PROB}")
print(f"  Referee:           {{1,2,3}} with P={{0.4, 0.2, 0.4}}")
print(f"  Known value:       v = −0.30")

In [ ]:
stats = train_agents(pi_agent, pii_agent,
                     num_episodes=NUM_EPISODES, log_interval=2000)

## 5. Visualising Training Progress

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Random Disclosure Parity Game — Gradient Bandit Training',
             fontsize=16, fontweight='bold', color='#e94560')
ep = stats['episodes']

# (0,0) Average payoff
ax = axes[0, 0]
window = 15
rolling = np.convolve(stats['avg_payoff_pi'],
                      np.ones(window)/window, mode='valid')
ax.plot(ep[window-1:], rolling, color='#f7b731', linewidth=1.2,
        label='Rolling avg payoff')
ax.axhline(y=-0.3, color='#e94560', linestyle='--', linewidth=1.5,
           label='Theory: v = −0.30')
ax.set_title('Average Payoff to PI', color='#e94560')
ax.set_xlabel('Episode')
ax.set_ylabel('Payoff')
ax.legend(fontsize=8, facecolor='#16213e', edgecolor='#4a4a6a')
ax.grid(True, alpha=0.3)

# (0,1) PI softmax probabilities
ax = axes[0, 1]
ax.plot(ep, stats['pi_prob_1'], color='#f7b731', linewidth=1,
        alpha=0.7, label='π(1)')
ax.plot(ep, stats['pi_prob_2'], color='#26de81', linewidth=1,
        alpha=0.7, label='π(2)')
ax.axhline(y=0.5, color='#e94560', linestyle='--', linewidth=1,
           alpha=0.6, label='Equilibrium: 0.50')
ax.set_title('Player I Policy π(a)', color='#e94560')
ax.set_xlabel('Episode')
ax.set_ylabel('Probability')
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=8, facecolor='#16213e', edgecolor='#4a4a6a')
ax.grid(True, alpha=0.3)

# (0,2) PI preferences
ax = axes[0, 2]
ax.plot(ep, stats['pi_H1'], color='#f7b731', linewidth=1,
        alpha=0.7, label='H(1)')
ax.plot(ep, stats['pi_H2'], color='#26de81', linewidth=1,
        alpha=0.7, label='H(2)')
ax.axhline(y=0, color='#666', linestyle=':', linewidth=0.5)
ax.set_title('Player I Preferences H(a)', color='#e94560')
ax.set_xlabel('Episode')
ax.set_ylabel('Preference')
ax.legend(fontsize=8, facecolor='#16213e', edgecolor='#4a4a6a')
ax.grid(True, alpha=0.3)

# (1,0) PII optimal-action probability at H1, H2
ax = axes[1, 0]
ax.plot(ep, stats['pii_H1_prob_3'], color='#f7b731', linewidth=1,
        alpha=0.7, label='π(3 | H₁)')
ax.plot(ep, stats['pii_H2_prob_4'], color='#26de81', linewidth=1,
        alpha=0.7, label='π(4 | H₂)')
ax.axhline(y=1.0, color='#e94560', linestyle='--', linewidth=1,
           alpha=0.5, label='Equilibrium: 1.0')
ax.set_title('PII Informed Policy (optimal action prob)',
             color='#e94560')
ax.set_xlabel('Episode')
ax.set_ylabel('Probability')
ax.set_ylim(-0.05, 1.1)
ax.legend(fontsize=8, facecolor='#16213e', edgecolor='#4a4a6a')
ax.grid(True, alpha=0.3)

# (1,1) PII at T — probability and preferences
ax = axes[1, 1]
ax.plot(ep, stats['pii_T_prob_3'], color='#a55eea', linewidth=1,
        alpha=0.7, label='π(3 | T)')
pii_T_prob_4 = [1 - p for p in stats['pii_T_prob_3']]
ax.plot(ep, pii_T_prob_4, color='#26de81', linewidth=1,
        alpha=0.7, label='π(4 | T)')
ax.axhline(y=0.5, color='#e94560', linestyle='--', linewidth=1,
           alpha=0.6, label='Equilibrium: 0.50')
ax.set_title('PII Uninformed Policy at T', color='#e94560')
ax.set_xlabel('Episode')
ax.set_ylabel('Probability')
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=8, facecolor='#16213e', edgecolor='#4a4a6a')
ax.grid(True, alpha=0.3)

# (1,2) Entropy of PI and PII at T
ax = axes[1, 2]
max_entropy = np.log(2)
ax.plot(ep, stats['entropy_pi'], color='#f7b731', linewidth=1,
        alpha=0.7, label='H(π_PI)')
ax.plot(ep, stats['entropy_pii_T'], color='#a55eea', linewidth=1,
        alpha=0.7, label='H(π_PII|T)')
ax.axhline(y=max_entropy, color='#e94560', linestyle='--',
           linewidth=1, alpha=0.5,
           label=f'Max entropy ({max_entropy:.3f})')
ax.set_title('Policy Entropy (exploration)', color='#e94560')
ax.set_xlabel('Episode')
ax.set_ylabel('Entropy (nats)')
ax.legend(fontsize=8, facecolor='#16213e', edgecolor='#4a4a6a')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Analysing Learned Strategies

We inspect the learned **preferences** $H(a)$ and the resulting **softmax probabilities** $\pi(a)$.

In [ ]:
print("═" * 60)
print("  LEARNED PREFERENCES AND POLICIES")
print("═" * 60)

print("\n── Player I (maximiser) ──")
pi_probs = pi_agent.get_probs()
print(f"  Preferences: H(1) = {pi_agent.H[0]:+.4f}, "
      f"H(2) = {pi_agent.H[1]:+.4f}")
print(f"  Policy:      π(1) = {pi_probs[1]:.4f}, "
      f"π(2) = {pi_probs[2]:.4f}")
print(f"  Equilibrium: π(1) = 0.5000, π(2) = 0.5000")
print(f"  Pref diff:   |H(1)−H(2)| = {abs(pi_agent.H[0]-pi_agent.H[1]):.4f}")

print("\n── Player II (minimiser) ──")
for info_set in ['H1', 'H2', 'T']:
    pii_probs = pii_agent.get_probs(info_set)
    h = pii_agent.H[info_set]
    print(f"\n  Info set {info_set}:")
    print(f"    Preferences: H(3) = {h[0]:+.4f}, H(4) = {h[1]:+.4f}")
    print(f"    Policy:      π(3) = {pii_probs[3]:.4f}, "
          f"π(4) = {pii_probs[4]:.4f}")

print("\n── Expected vs Learned ──")
print("  H1: optimal π(3)=1.0 → "
      f"learned π(3)={pii_agent.get_probs('H1')[3]:.4f}")
print("  H2: optimal π(4)=1.0 → "
      f"learned π(4)={pii_agent.get_probs('H2')[4]:.4f}")
print("  T:  optimal π(3)=0.5 → "
      f"learned π(3)={pii_agent.get_probs('T')[3]:.4f}")
print(f"  PI: optimal π(1)=0.5 → "
      f"learned π(1)={pi_agent.get_probs()[1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle('Learned Policies — Gradient Bandit',
             fontsize=15, fontweight='bold', color='#e94560')
colors = ['#f7b731', '#26de81']

# Player I
ax = axes[0]
probs = pi_agent.get_probs()
vals = [probs[a] for a in PI_ACTIONS]
bars = ax.bar([str(a) for a in PI_ACTIONS], vals,
              color=colors, edgecolor='#4a4a6a')
ax.set_title('Player I π(a)', color='#e94560', fontweight='bold')
ax.set_xlabel('Action')
ax.set_ylabel('Probability')
ax.axhline(y=0.5, color='#e94560', linestyle='--', linewidth=1,
           label='Equil: 0.50')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=8, facecolor='#16213e', edgecolor='#4a4a6a')
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.02,
            f'{v:.3f}', ha='center', va='bottom', fontsize=10)

# PII at each info set
for idx, info_set in enumerate(['H1', 'H2', 'T']):
    ax = axes[idx + 1]
    probs = pii_agent.get_probs(info_set)
    vals = [probs[a] for a in PII_ACTIONS]
    bars = ax.bar([str(a) for a in PII_ACTIONS], vals,
                  color=colors, edgecolor='#4a4a6a')
    ax.set_title(f'PII π(a | {info_set})',
                 color='#e94560', fontweight='bold')
    ax.set_xlabel('Action')
    ax.set_ylabel('Probability')
    ax.set_ylim(0, 1.05)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.02,
                f'{v:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## 7. Theoretical Optimal Strategy

From the minimax analysis (see `src/minimax/random_disclosure_parity_game_demo.py`), the conditional payoffs to Player I are:

| $g(i,j)$ | $j=3$ | $j=4$ |
|:---------:|:-----:|:-----:|
| $i=1$ | $-0.6$ | $+0.6$ |
| $i=2$ | $+0.6$ | $-0.6$ |

Player II's optimal strategy:
- **$H_1$**: play 3 → $g(1,3) = -0.6$
- **$H_2$**: play 4 → $g(2,4) = -0.6$
- **$T$**: randomise 50/50 → expected payoff = 0

Game value: $v = \frac{1}{2}(-0.6) + \frac{1}{2}(0) = -0.30$

In [ ]:
def g(i: int, j: int) -> float:
    """Conditional payoff to PI for choices (i, j), averaged over referee."""
    return sum(p * (1 if (i + j + r) % 2 == 0 else -1)
               for r, p in REFEREE_OUTCOMES)


print("═" * 60)
print("  COMPARISON: LEARNED vs THEORETICAL OPTIMAL")
print("═" * 60)

print("\n── Conditional payoffs g(i, j) ──")
print(f"  g(1,3) = {g(1,3):+.1f}    g(1,4) = {g(1,4):+.1f}")
print(f"  g(2,3) = {g(2,3):+.1f}    g(2,4) = {g(2,4):+.1f}")

print("\n── Player II ──")
optimal_pii = {'H1': (3, 1.0), 'H2': (4, 1.0), 'T': ('mix', 0.5)}
for info_set in ['H1', 'H2', 'T']:
    learned_probs = pii_agent.get_probs(info_set)
    opt_action, opt_prob = optimal_pii[info_set]
    if info_set == 'T':
        ok = abs(learned_probs[3] - 0.5) < 0.15
        print(f"  {info_set}: π(3) = {learned_probs[3]:.4f}  "
              f"(optimal: 0.5000)  {'✓' if ok else '~'}")
    else:
        ok = learned_probs[opt_action] > 0.85
        print(f"  {info_set}: π({opt_action}) = {learned_probs[opt_action]:.4f}  "
              f"(optimal: {opt_prob:.1f})  {'✓' if ok else '~'}")

print("\n── Player I ──")
pi_probs = pi_agent.get_probs()
ok_pi = abs(pi_probs[1] - 0.5) < 0.15
print(f"  π(1) = {pi_probs[1]:.4f}  (optimal: 0.5000)  {'✓' if ok_pi else '~'}")

## 8. Evaluation: Learned Agents vs Optimal Play

In [ ]:
class OptimalPIAgent:
    """Optimal PI: mixes {1, 2} uniformly."""
    def select_action(self, training=False) -> int:
        return random.choice(PI_ACTIONS)


class OptimalPIIAgent:
    """Optimal PII: match parity when informed, randomise when not."""
    def select_action(self, info_set: str, training=False) -> int:
        if info_set == 'H1': return 3
        if info_set == 'H2': return 4
        return random.choice(PII_ACTIONS)


def evaluate(pi_fn, pii_fn, num_games: int = 100_000) -> float:
    """Play num_games, return average payoff to PI."""
    game = ParityGame()
    total = 0.0
    for _ in range(num_games):
        game.reset()
        i = pi_fn()
        game.pi_step(i)
        info = game.get_pii_info_set()
        j = pii_fn(info)
        game.pii_step(j)
        total += game.payoff_pi
    return total / num_games


random.seed(123)
np.random.seed(123)
opt_pi = OptimalPIAgent()
opt_pii = OptimalPIIAgent()

print("═" * 60)
print("  EVALUATION (100,000 games per matchup)")
print("═" * 60)

matchups = [
    ('Learned PI vs Learned PII',
     lambda: pi_agent.select_action(False),
     lambda s: pii_agent.select_action(s, False)),
    ('Learned PI vs Optimal PII',
     lambda: pi_agent.select_action(False),
     lambda s: opt_pii.select_action(s)),
    ('Optimal PI vs Learned PII',
     lambda: opt_pi.select_action(),
     lambda s: pii_agent.select_action(s, False)),
    ('Optimal PI vs Optimal PII',
     lambda: opt_pi.select_action(),
     lambda s: opt_pii.select_action(s)),
]

results = []
for label, pi_fn, pii_fn in matchups:
    avg = evaluate(pi_fn, pii_fn)
    results.append((label, avg))
    print(f"  {label:>40s}: {avg:+.4f}")

print(f"\n  Theoretical game value: −0.3000")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

labels = [r[0] for r in results]
values = [r[1] for r in results]
bar_colors = ['#f7b731' if v > -0.3 else '#26de81' for v in values]

bars = ax.barh(labels, values, color=bar_colors,
               edgecolor='#4a4a6a', height=0.5)
ax.axvline(x=-0.3, color='#e94560', linestyle='--', linewidth=2,
           label='Game value (−0.30)')
ax.axvline(x=0, color='#4a4a6a', linestyle='-', linewidth=0.5)
ax.set_xlabel('Average Payoff to Player I ($)')
ax.set_title('Evaluation: Learned vs Optimal Agents',
             fontsize=14, fontweight='bold', color='#e94560')
ax.legend(loc='lower right',
          facecolor='#16213e', edgecolor='#4a4a6a')
ax.set_xlim(-0.7, 0.15)

for bar, v in zip(bars, values):
    offset = -0.02 if v < 0 else 0.02
    ax.text(v + offset, bar.get_y() + bar.get_height()/2,
            f'{v:+.3f}', ha='right' if v < 0 else 'left',
            va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 9. Watching Sample Games

In [ ]:
def play_verbose(pi_fn, pii_fn, pi_name='PI', pii_name='PII',
                 pi_probs_fn=None, pii_probs_fn=None, n=10):
    """Play n games with detailed output."""
    game = ParityGame()
    for k in range(n):
        game.reset()
        i = pi_fn()
        game.pi_step(i)
        info = game.get_pii_info_set()
        j = pii_fn(info)
        game.pii_step(j)
        s = game.pi_choice + game.pii_choice + game.ref_draw
        par = 'even' if s % 2 == 0 else 'odd'
        winner = pi_name if game.payoff_pi > 0 else pii_name

        extra = ''
        if pi_probs_fn:
            pp = pi_probs_fn()
            extra += f' π_PI=[{pp[1]:.2f},{pp[2]:.2f}]'
        if pii_probs_fn:
            qp = pii_probs_fn(info)
            extra += f' π_PII|{info}=[{qp[3]:.2f},{qp[4]:.2f}]'

        print(f"  Game {k+1:2d}: {pi_name}={i}, coin={game.coin}, "
              f"info={info}, {pii_name}={j}, r={game.ref_draw}, "
              f"sum={s} ({par}) → {winner} wins{extra}")


random.seed(99)
np.random.seed(99)
print("═" * 75)
print("  SAMPLE GAMES: Learned PI vs Learned PII")
print("═" * 75)
play_verbose(
    lambda: pi_agent.select_action(False),
    lambda s: pii_agent.select_action(s, False),
    'Learned PI', 'Learned PII',
    pi_probs_fn=lambda: pi_agent.get_probs(),
    pii_probs_fn=lambda s: pii_agent.get_probs(s))

random.seed(99)
np.random.seed(99)
print(f"\n{'═' * 75}")
print("  SAMPLE GAMES: Optimal PI vs Optimal PII")
print("═" * 75)
play_verbose(
    lambda: opt_pi.select_action(),
    lambda s: opt_pii.select_action(s),
    'Optimal PI', 'Optimal PII')

## 10. Ablation: Baseline, Decay, and Learning Rate

We compare convergence under different hyperparameter settings.

In [ ]:
configs = [
    {'alpha': 0.05, 'decay': 0.001, 'baseline': True,
     'label': 'α=0.05, λ=0.001, baseline'},
    {'alpha': 0.05, 'decay': 0.001, 'baseline': False,
     'label': 'α=0.05, λ=0.001, no baseline'},
    {'alpha': 0.05, 'decay': 0.0001, 'baseline': True,
     'label': 'α=0.05, λ=0.0001, baseline'},
    {'alpha': 0.15, 'decay': 0.001, 'baseline': True,
     'label': 'α=0.15, λ=0.001, baseline'},
]

ablation_results = []

for cfg in configs:
    np.random.seed(42)
    random.seed(42)
    pa = GradientBanditPI(alpha=cfg['alpha'], decay=cfg['decay'],
                          use_baseline=cfg['baseline'])
    pb = GradientBanditPII(alpha=cfg['alpha'], decay=cfg['decay'],
                           use_baseline=cfg['baseline'])
    st = train_agents(pa, pb, num_episodes=300_000, log_interval=2000)
    ablation_results.append((cfg['label'], st, pa, pb))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Ablation: Baseline, Decay & Learning Rate',
             fontsize=15, fontweight='bold', color='#e94560')
cmap = ['#f7b731', '#26de81', '#a55eea', '#e94560']

# Average payoff convergence
ax = axes[0]
for (label, st, _, _), col in zip(ablation_results, cmap):
    w = 15
    roll = np.convolve(st['avg_payoff_pi'],
                       np.ones(w)/w, mode='valid')
    ax.plot(st['episodes'][w-1:], roll, color=col,
            linewidth=1.2, alpha=0.8, label=label)
ax.axhline(y=-0.3, color='white', linestyle='--', linewidth=1.5,
           alpha=0.5, label='v* = −0.30')
ax.set_xlabel('Episode')
ax.set_ylabel('Avg Payoff to PI')
ax.set_title('Payoff Convergence', color='#e94560')
ax.legend(fontsize=7, facecolor='#16213e', edgecolor='#4a4a6a')
ax.grid(True, alpha=0.3)

# PI mixing quality
ax = axes[1]
for (label, st, _, _), col in zip(ablation_results, cmap):
    w = 15
    roll = np.convolve(st['pi_prob_1'],
                       np.ones(w)/w, mode='valid')
    ax.plot(st['episodes'][w-1:], roll, color=col,
            linewidth=1.2, alpha=0.8, label=label)
ax.axhline(y=0.5, color='white', linestyle='--', linewidth=1.5,
           alpha=0.5, label='Equilibrium: 0.50')
ax.set_xlabel('Episode')
ax.set_ylabel('π(1)')
ax.set_title('PI Mixing at Equilibrium', color='#e94560')
ax.set_ylim(0, 1)
ax.legend(fontsize=7, facecolor='#16213e', edgecolor='#4a4a6a')
ax.grid(True, alpha=0.3)

# PII T mixing quality
ax = axes[2]
for (label, st, _, _), col in zip(ablation_results, cmap):
    w = 15
    roll = np.convolve(st['pii_T_prob_3'],
                       np.ones(w)/w, mode='valid')
    ax.plot(st['episodes'][w-1:], roll, color=col,
            linewidth=1.2, alpha=0.8, label=label)
ax.axhline(y=0.5, color='white', linestyle='--', linewidth=1.5,
           alpha=0.5, label='Equilibrium: 0.50')
ax.set_xlabel('Episode')
ax.set_ylabel('π(3 | T)')
ax.set_title('PII Mixing at T', color='#e94560')
ax.set_ylim(0, 1)
ax.legend(fontsize=7, facecolor='#16213e', edgecolor='#4a4a6a')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Conclusion

### Summary

We trained two **Gradient Bandit** agents (with preference decay) on the Random Disclosure Parity Game — a zero-sum game with imperfect information and chance moves.

**Key findings:**

1. **Player II's informed strategy**: Preferences at $H_1$ and $H_2$ diverge cleanly — the agent learns to **match parity** (play 3 when knowing PI chose 1, play 4 when knowing PI chose 2). The softmax probabilities converge to near 1.0 for the correct action.

2. **Player II's uninformed strategy**: At information set $T$, both actions yield the same expected payoff when PI mixes uniformly. The preference decay prevents divergence, and the softmax output hovers around **50/50** — the optimal randomisation.

3. **Player I's indifference**: Both preferences oscillate near zero thanks to the decay. The softmax policy maintains approximately **50/50 mixing**, reflecting the theoretical result that PI is indifferent at equilibrium.

4. **Game value**: The average payoff converges to approximately **$-0.30$**, matching $v = -0.6 \times P(\text{heads}) = -0.30$.

### Preference Decay: Stabilising Adversarial Self-Play

Without preference decay, gradient bandits in self-play suffer from **preference divergence**: each player's preferences spiral to $\pm\infty$ as they chase the other's shifting strategy. The decay term $H \leftarrow (1-\lambda)H$ acts as a regulariser that:

- **Prevents divergence** at decision points requiring mixing ($T$, PI) by pulling preferences toward zero
- **Preserves strong signals** at deterministic info sets ($H_1$, $H_2$), where the gradient updates consistently reinforce the correct action faster than decay erodes it
- **Balances exploration and exploitation** by keeping the policy entropy near the equilibrium level

### Gradient Bandit vs Q-Learning for This Game

| Aspect | Gradient Bandit | Q-Learning (softmax) |
|:-------|:---------------|:--------------------|
| What it learns | Action *preferences* H(a) | Action *values* Q(a) |
| Policy | Softmax on preferences | Softmax on Q-values (fixed τ) |
| Update rule | Policy gradient (REINFORCE) | Bellman equation |
| Mixed strategies | Natural: equal H → 50/50 | Requires fixed temperature τ |
| Stabilisation | Preference decay λ | Empirical freq averaging |
| Baseline | Reduces variance | Not applicable |

### Connections to Blackwell's Work

The gradient bandit update is a special case of **stochastic approximation** — a field pioneered by Blackwell and others. The preference-based policy gradient can be viewed as a form of Blackwell's *approachability*, where the agent's time-averaged strategy converges to the equilibrium payoff set. The preference decay ensures that the sequence of play remains in the approachable set.